In [0]:

CREATE OR REPLACE TABLE
telecom_lakehouse.bronze.subscriptions_raw
USING DELTA
COMMENT 'Synthetic telecom customer subscription source'
AS

SELECT

    CONCAT(
        'SUB',
        LPAD(CAST(s.id AS STRING), 8, '0')
    ) AS subscription_id,

    CONCAT(
        'CUST',
        LPAD(
            CAST(PMOD(s.id * 37, 10000) + 1 AS STRING),
            7,
            '0'
        )
    ) AS customer_id,

    CONCAT(
        'PROD',
        LPAD(
            CAST(PMOD(s.id * 13, 500) + 1 AS STRING),
            4,
            '0'
        )
    ) AS product_id,

    CASE
        WHEN PMOD(s.id, 10) < 7 THEN 'ACTIVE'
        WHEN PMOD(s.id, 10) < 9 THEN 'SUSPENDED'
        ELSE 'TERMINATED'
    END AS subscription_status,

    p.product_type,

    p.monthly_price,

    DATE_ADD(
        DATE'2023-01-01',
        CAST(PMOD(s.id * 19, 1000) AS INT)
    ) AS subscription_start_date,

    CASE
        WHEN PMOD(s.id, 10) = 9
        THEN DATE_ADD(
            DATE'2023-01-01',
            CAST(PMOD(s.id * 19, 1000) + 365 AS INT)
        )
        ELSE NULL
    END AS subscription_end_date,

    CASE
        WHEN PMOD(s.id, 2) = 0 THEN 'MONTHLY'
        ELSE 'ANNUAL'
    END AS billing_frequency,

    'SUBSCRIPTION_SYSTEM' AS source_system,

    'BATCH_001' AS ingest_batch_id,

    CURRENT_TIMESTAMP() AS ingestion_timestamp

FROM RANGE(1, 100001) s

INNER JOIN telecom_lakehouse.bronze.products_raw p


ON p.product_id = CONCAT(
    'PROD',
    LPAD(
        CAST(PMOD(s.id * 13, 500) + 1 AS STRING),
        4,
        '0'
    )
);
